# Project 4: MPTE Investigation — Summary of Findings

This notebook summarizes the investigation of the Mixed-Panels-Transformer Encoder
(Brini & Seregina, 2026) across multiple financial applications.

**Paper**: arXiv:2601.16274v1 — *A Nonlinear Target-Factor Model with Attention Mechanism for Mixed-Frequency Data*

**Source**: Presented by Alessio Brini (Duke) at the Quaint Quant Conference, SMU Cox School of Business.

### Bottom line
The MPTE transformer does not outperform simpler models (AR, Ridge, XGB, PCA) in any
financial application tested. The paper's results hold for macro-to-macro forecasting
but do not extend to financial targets at this sample size (~120-480 training samples).

## 1. MPTE Architecture

The model processes mixed-frequency data as a token sequence:
- Each observation = (standardized value, learned variable embedding, learned frequency embedding)
- Sinusoidal temporal encoding for irregular time spacing
- Transformer encoder with self-attention
- Mean-pooled representation → prediction head

Our implementation: d_model=32, nhead=2, 1 layer, d_ff=64, ~10K parameters.
Smaller than the paper's config to match our smaller training sets.

## 2. Results Across All Applications

| # | Application | MPTE | Best Simple Model | Winner | Notebooks |
|---|---|---|---|---|---|
| 1 | Vol forecasting (RMSE) | 0.0850 | AR 0.0807 | AR | 2, 3 |
| 2 | Risk filter (Sharpe) | 0.79 | AR 0.89 | AR | 5 |
| 3 | Cross-asset ranking Q (Top-1) | 30.4% | Ridge 37.5% | Ridge | 4 |
| 4 | Cross-asset ranking M (Top-1) | 25.7% | Ridge 34.1% | Ridge | 6 |
| 5 | Factor ranking (Top-1) | 30.3%* | XGB 27.3% | MPTE* | 7 |
| 6 | Regime identification (Sharpe) | 0.14 | PCA+GMM 0.32 | PCA+GMM | 9 |
| 7 | Enriched panel vol (RMSE) | 0.0879 | AR 0.0807 | AR | 10 |
| 8 | Enriched panel regime (Sharpe) | 0.18 | PCA+GMM 0.35 | PCA+GMM | 10 |

*MPTE won factor ranking via degenerate solution (predicted Momentum as #1 for all 165 months).

**MPTE never won a clean comparison against any simple model.**

## 3. Why MPTE Underperforms

### Sample size
The paper uses ~260 quarterly observations (1959-2025) with the full FRED-MD panel (135 variables).
We had 120-480 training samples depending on application and frequency, with 34-47 variables.
Transformers are data-hungry — 10K parameters still needs substantial training data to learn
meaningful attention patterns.

### Target mismatch
The paper predicts macro variables (GDP, CPI, unemployment) FROM other macro variables.
The target Y is part of the same panel as inputs X. The attention mechanism learns
cross-variable relationships within the macro system.

We predicted financial targets (vol, returns, factor returns) that were NOT in the panel.
Adding them to the panel (notebook 10) helped slightly but didn't close the gap.
The macro→financial mapping may simply be too weak for attention to capture.

### Signal-to-noise
Financial returns have much lower predictability than macro aggregates.
GDP growth has R² of 0.3-0.5 from macro variables; stock returns have R² of 0.01-0.05.
The attention mechanism needs clean signal to learn useful patterns.

### Hyperparameter tuning
The paper runs 500 Optuna trials per target. We used fixed defaults.
However, if the architecture can't beat Ridge with reasonable defaults,
tuning is unlikely to produce a qualitative change.

## 4. What DID Work

### PCA+GMM regime conditioning (Project 3)
- 0.36 Sharpe vs 0.11 equal-weight, walk-forward
- 3 economically interpretable regimes
- Value-add in all regimes
- No neural networks needed

### Vol timing theoretical value (Project 2)
- Perfect foresight filter: 1.49 Sharpe vs 0.92 buy & hold
- Saves 43% of losses during worst quarters
- Gap between perfect and real forecast = research opportunity

### Ridge-based relative ranking
- 34% top-1 accuracy for asset classes (vs 25% random)
- Positive correlations on individual assets
- Linear models capture the available signal

## 5. Lessons Learned

1. **Start simple.** AR, Ridge, and PCA beat or match the transformer everywhere.
   Always establish strong baselines before adding complexity.

2. **Match the method to the data.** 10K-parameter transformers need more than
   120 training samples. With small data, linear models are hard to beat.

3. **Regime classification > return prediction.** Classifying the current macro state
   (easier, persistent) works better than predicting next-period returns (hard, noisy).

4. **Panel composition matters.** The paper puts targets in the panel;
   we initially didn't. This was a design error, though fixing it didn't
   change the conclusion.

5. **Negative results are valuable.** Knowing that MPTE doesn't extend to
   financial forecasting saves future research time.

6. **Factor timing is hard.** Consistent with Asness (2016, "The Siren Song of Factor Timing").
   Regime-based tilting works; month-to-month timing doesn't.

## 6. Original Notebooks Reference

| Notebook | Content | Status |
|---|---|---|
| 01 | FRED data acquisition + WRDS | → Project 1 |
| 02 | MPTE single-split vol model | Archived (superseded by 03) |
| 03 | Walk-forward vol forecasting | → Project 2 Part A |
| 04 | Cross-asset quarterly ranking | Archived (superseded by 06) |
| 05 | Risk filter stress test | → Project 2 Part B |
| 06 | Monthly relative asset ranking | Archived (superseded by factor work) |
| 07 | Fama-French factor timing | Archived (degenerate MPTE result) |
| 08 | PCA+GMM regime factors (3 regimes) | → **Project 3** (deployable) |
| 09 | MPTE+GMM regime factors | Archived (superseded by 10) |
| 10 | Enriched panel (macro + financial) | → Project 4 (this summary) |

## 7. Future Research Directions

### High priority
- Extend regime framework (Project 3) to more asset classes, sectors, or international factors
- Test with transaction costs and realistic rebalancing constraints
- Try different clustering methods (spectral, hierarchical) for regime identification

### Medium priority
- Better vol models using daily data, implied vol surfaces, credit spreads (Project 2)
- Monthly vol forecasting instead of quarterly
- MPTE with much larger dataset (full FRED-MD, 135 variables, 1959-present)

### Low priority
- Optuna hyperparameter tuning for MPTE
- Attention weight analysis (interpretability)
- MPTE for macro-to-macro forecasting (the paper's actual use case)